# RL Derivative Hedging - Mathematical Foundations

This notebook explores the financial and mathematical concepts underlying the project. It does not run training or load a trained RL policy.

Contents:
1. Black-Scholes pricing
2. Greeks and their intuition
3. GBM simulation
4. Delta hedging mechanics
5. Why discrete hedging leaves residual P&L
6. Transaction cost impact


## Setup

Load the project configuration and modules so examples use the same default market parameters as the training and evaluation pipeline.

In [1]:
from __future__ import annotations

import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if "." not in sys.path:
    sys.path.insert(0, ".")

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from src.finance.black_scholes import call_delta, call_price, gamma, put_price
from src.finance.transaction_costs import proportional_cost
from src.simulation.gbm import GBMSimulator
from src.training.hyperparams import load_config

pio.renderers.default = "plotly_mimetype"
config = load_config("configs/training.yaml")
env = config.eval_environment
S0, K, r, sigma, dt, n_steps, kappa = (
    env.S0,
    env.K,
    env.r,
    env.sigma,
    env.dt,
    env.T,
    env.kappa,
)
T_years = n_steps * dt
print(f"Loaded config: S0={S0:.1f}, K={K:.1f}, r={r:.3f}, sigma={sigma:.3f}, steps={n_steps}, kappa={kappa:.4f}")

Loaded config: S0=100.0, K=100.0, r=0.050, sigma=0.200, steps=30, kappa=0.0010


## 1. Black-Scholes Pricing

The call value increases with spot price. The put value decreases with spot price. This cell plots both across moneyness levels around the configured strike.

In [2]:
spots = np.linspace(0.6 * K, 1.4 * K, 81)
call_values = call_price(spots, K, r, sigma, T_years)
put_values = put_price(spots, K, r, sigma, T_years)

fig = go.Figure()
fig.add_trace(go.Scatter(x=spots, y=call_values, name="Call price"))
fig.add_trace(go.Scatter(x=spots, y=put_values, name="Put price"))
fig.add_vline(x=K, line_dash="dash", annotation_text="Strike")
fig.update_layout(title="Black-Scholes option prices", xaxis_title="Spot price", yaxis_title="Option value")
print(f"ATM call={float(call_price(S0, K, r, sigma, T_years)):.4f}; ATM put={float(put_price(S0, K, r, sigma, T_years)):.4f}")
fig.show()

ATM call=3.0511; ATM put=2.4576


Put-call parity is a useful consistency check: `call + discounted strike = put + spot`. The maximum error should be near floating-point precision.

In [3]:
parity_error = call_values + K * np.exp(-r * T_years) - put_values - spots
print(f"Max absolute put-call parity error: {np.max(np.abs(parity_error)):.12f}")

Max absolute put-call parity error: 0.000000000000


## 2. Greeks

Delta measures first-order sensitivity to the underlying, while gamma measures how quickly delta changes. Gamma peaks near-the-money, where hedging is hardest.

In [4]:
deltas = call_delta(spots, K, r, sigma, T_years)
gammas = gamma(spots, K, r, sigma, T_years)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Call delta", "Gamma"))
fig.add_trace(go.Scatter(x=spots, y=deltas, name="Delta"), row=1, col=1)
fig.add_trace(go.Scatter(x=spots, y=gammas, name="Gamma"), row=2, col=1)
fig.add_vline(x=K, line_dash="dash")
fig.update_layout(title="Greeks across moneyness", showlegend=False)
fig.update_xaxes(title_text="Spot price", row=2, col=1)
print(f"Peak gamma in grid: {float(np.max(gammas)):.6f}")
fig.show()

Peak gamma in grid: 0.058380


As expiry approaches, call delta approaches an intrinsic-value step function: close to 0 below the strike and close to 1 above it.

In [5]:
time_grid = np.array([T_years, T_years / 2, max(dt, T_years / 10), dt])
fig = go.Figure()
for tau in time_grid:
    fig.add_trace(go.Scatter(x=spots, y=call_delta(spots, K, r, sigma, tau), name=f"T={tau:.3f}y"))
fig.add_vline(x=K, line_dash="dash", annotation_text="Strike")
fig.update_layout(title="Delta sharpens near expiry", xaxis_title="Spot price", yaxis_title="Call delta")
print(f"Displayed {len(time_grid)} time-to-expiry slices")
fig.show()

Displayed 4 time-to-expiry slices


## 3. GBM Simulation

The environment uses Geometric Brownian Motion paths. This plot shows a sample of 50 paths generated with the configured market parameters.

In [6]:
sim = GBMSimulator(S0=S0, mu=r, sigma=sigma, dt=dt, seed=config.run.seed)
paths = sim.generate_paths(n_paths=50, n_steps=n_steps)
steps = np.arange(n_steps + 1)

fig = go.Figure()
for path in paths:
    fig.add_trace(go.Scatter(x=steps, y=path, mode="lines", line=dict(width=1), showlegend=False))
fig.update_layout(title="Sample GBM price paths", xaxis_title="Step", yaxis_title="Spot price")
print(f"Generated path matrix shape: {paths.shape}")
fig.show()

Generated path matrix shape: (50, 31)


The terminal distribution of GBM is lognormal. The sample terminal mean should be close to the analytical mean `S0 * exp(rT)`.

In [7]:
dist_sim = GBMSimulator(S0=S0, mu=r, sigma=sigma, dt=dt, seed=config.run.seed + 1)
dist_paths = dist_sim.generate_paths(n_paths=5000, n_steps=n_steps)
terminal = dist_paths[:, -1]
theory_mean = S0 * np.exp(r * T_years)
theory_var = S0**2 * np.exp(2 * r * T_years) * (np.exp(sigma**2 * T_years) - 1)

fig = go.Figure(data=[go.Histogram(x=terminal, nbinsx=60, histnorm="probability density")])
fig.add_vline(x=theory_mean, line_dash="dash", annotation_text="Theoretical mean")
fig.update_layout(title="Terminal GBM price distribution", xaxis_title="Terminal spot", yaxis_title="Density")
print(f"Sample mean={np.mean(terminal):.4f}; theoretical mean={theory_mean:.4f}; theoretical variance={theory_var:.4f}")
fig.show()

Sample mean=100.4864; theoretical mean=100.5970; theoretical variance=48.3011


Higher volatility widens the path distribution. The same seed is used for each volatility level so the visual comparison is stable.

In [8]:
sigma_levels = [env.sigma_range[0], sigma, env.sigma_range[1]]
fig = go.Figure()
for sigma_level in sigma_levels:
    sigma_sim = GBMSimulator(S0=S0, mu=r, sigma=sigma_level, dt=dt, seed=config.run.seed + 2)
    sigma_paths = sigma_sim.generate_paths(n_paths=12, n_steps=n_steps)
    mean_path = sigma_paths.mean(axis=0)
    fig.add_trace(go.Scatter(x=steps, y=mean_path, name=f"sigma={sigma_level:.2f}"))
fig.update_layout(title="Average path by volatility level", xaxis_title="Step", yaxis_title="Mean spot")
print(f"Compared volatility levels: {', '.join(f'{x:.2f}' for x in sigma_levels)}")
fig.show()

Compared volatility levels: 0.10, 0.20, 0.40


## 4. Delta Hedging Mechanics

This cell manually simulates one Black-Scholes delta-hedged episode. It does not call the Gymnasium environment, which makes the accounting transparent.

In [9]:
one_path = GBMSimulator(S0=S0, mu=r, sigma=sigma, dt=dt, seed=config.run.seed + 3).generate_path(n_steps)
hedge_pos = 0.0
cumulative_pnl = 0.0
cumulative_cost = 0.0
hedges = []
pnl_path = []
cost_path = []

for step in range(n_steps):
    tau_prev = max((n_steps - step) * dt, 1e-8)
    tau_next = max((n_steps - step - 1) * dt, 0.0)
    target_hedge = float(call_delta(one_path[step], K, r, sigma, tau_prev))
    trade = target_hedge - hedge_pos
    cost = float(proportional_cost(trade, one_path[step], kappa))
    option_prev = float(call_price(one_path[step], K, r, sigma, tau_prev))
    if step == n_steps - 1:
        option_next = float(max(one_path[step + 1] - K, 0.0))
    else:
        option_next = float(call_price(one_path[step + 1], K, r, sigma, tau_next))
    step_pnl = (option_prev - option_next) + target_hedge * (one_path[step + 1] - one_path[step]) - cost
    cumulative_pnl += step_pnl
    cumulative_cost += cost
    hedge_pos = target_hedge
    hedges.append(target_hedge)
    pnl_path.append(cumulative_pnl)
    cost_path.append(cumulative_cost)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Spot", "BS hedge ratio", "Cumulative P&L and cost"))
fig.add_trace(go.Scatter(x=steps, y=one_path, name="Spot"), row=1, col=1)
fig.add_trace(go.Scatter(x=steps[1:], y=hedges, name="Delta hedge"), row=2, col=1)
fig.add_trace(go.Scatter(x=steps[1:], y=pnl_path, name="Cumulative P&L"), row=3, col=1)
fig.add_trace(go.Scatter(x=steps[1:], y=cost_path, name="Cumulative cost"), row=3, col=1)
fig.update_layout(title="Manual discrete delta-hedging episode")
print(f"Terminal hedged P&L={cumulative_pnl:.4f}; total transaction cost={cumulative_cost:.4f}")
fig.show()

Terminal hedged P&L=0.5817; total transaction cost=0.1858


Discrete hedging leaves residual P&L because delta is only updated at each step, not continuously. The residual is larger when price moves sharply between rebalances.

In [10]:
price_moves = np.diff(one_path)
largest_move_step = int(np.argmax(np.abs(price_moves))) + 1
largest_move = float(price_moves[largest_move_step - 1])
print(f"Largest one-step price move occurred at step {largest_move_step}: {largest_move:.4f}")
print(f"Residual terminal P&L after costs: {cumulative_pnl:.4f}")

Largest one-step price move occurred at step 27: -2.1375
Residual terminal P&L after costs: 0.5817


## 5. Transaction Cost Impact

A proportional transaction cost penalises changes in hedge ratio. Higher `kappa` makes frequent rebalancing more expensive.

In [11]:
kappa_levels = [0.0, kappa, 2 * kappa, 5 * kappa, 10 * kappa]
fig = go.Figure()
final_costs = []
for kappa_level in kappa_levels:
    hedge_pos = 0.0
    running_cost = 0.0
    running_costs = []
    for step in range(n_steps):
        tau_prev = max((n_steps - step) * dt, 1e-8)
        target_hedge = float(call_delta(one_path[step], K, r, sigma, tau_prev))
        trade = target_hedge - hedge_pos
        running_cost += float(proportional_cost(trade, one_path[step], kappa_level))
        hedge_pos = target_hedge
        running_costs.append(running_cost)
    final_costs.append(running_cost)
    fig.add_trace(go.Scatter(x=steps[1:], y=running_costs, name=f"kappa={kappa_level:.4f}"))
fig.update_layout(title="Cumulative transaction cost by kappa", xaxis_title="Step", yaxis_title="Cumulative cost")
print("Final costs: " + ", ".join(f"{cost:.4f}" for cost in final_costs))
fig.show()

Final costs: 0.0000, 0.1858, 0.3716, 0.9289, 1.8578


The RL task is to learn this trade-off directly: hedge enough to control risk, but avoid unnecessary turnover when transaction costs are material.

In [12]:
baseline_cost = final_costs[1]
high_cost = final_costs[-1]
print(f"At 10x default kappa, this path's BS-delta transaction cost changes from {baseline_cost:.4f} to {high_cost:.4f}")

At 10x default kappa, this path's BS-delta transaction cost changes from 0.1858 to 1.8578
